# 🎓 Modelo de Machine Learning - Previsão de Evasão (Aperfeiçoamento / Extensão)
### a instituicao | Versão v3 (Foco em Alto Recall & Predição Financeiro-Comportamental)

Este notebook implementa o pipeline de Machine Learning para cursos de **Aperfeiçoamento e Extensão (`EJND_EXT`)** utilizando a base de dados institucional do ERP academico.

#### 🎯 Particularidades e Adaptações Desta Modalidade:
1. **Natureza de Ciclo Único:** Cursos livres e de aperfeiçoamento não possuem rematrícula modular ano a ano; a evasão é definida pelo **cancelamento/desistência** formal no curso (`COL_SITUACAO IN ('Cancelado', 'Evadido', 'Inconcluido')`).
2. **Ausência de Notas Fracionadas:** Por não haver diário de classe tradicional fracionado por bimestre, as colunas acadêmicas e de faltas foram limpas.
3. **Forte Sinal Preditivo Financeiro:** A inadimplência no pagamento da taxa/parcelas do curso e o histórico de atraso são as variáveis centrais.
4. **Adesão ao AVA:** A integração com o AVA-web/AVA-app é tratada de forma suave via `LEFT JOIN` para não descartar alunos presenciais.
5. **Foco Operacional em Alto Recall:** Otimização do limiar de probabilidade para garantir identificação e resgate de $\ge 85\%$ dos desistentes.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
    f1_score,
    recall_score,
    precision_score
)

# Configuracao visual
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
print('✅ Bibliotecas importadas com sucesso!')

## 1. Carregamento dos Dados
Lemos a base extraída para os cursos de Aperfeiçoamento/Extensão (`tabelas/extracao_extensao_2025_5.xlsx`).

In [ ]:
# Caminho relativo flexivel para o arquivo de dados
candidate_paths = [
    'tabelas/extracao_extensao_2025_5.xlsx',
    os.path.join('..', '..', 'tabelas', 'extracao_extensao_2025_5.xlsx'),
    os.path.join('..', 'tabelas', 'extracao_extensao_2025_5.xlsx'),
    'C:/Users/leonardo.marques/Documents/machine-learning-evasao/tabelas/extracao_extensao_2025_5.xlsx'
]
file_path = next((p for p in candidate_paths if os.path.exists(p)), candidate_paths[-1])

print(f'Carregando dados de: {file_path}')
df = pd.read_excel(file_path)
print(f'Dimensões brutas: {df.shape[0]} linhas x {df.shape[1]} colunas')
print(df[['ALUNO', 'CURSO', 'NOME_CURSO', 'ANO_INGRESSO', 'SIT_ALVO', 'EVADIU']].head(3))

## 2. Filtro Obrigatório de Elegibilidade (`ELEGIVEL == 1`)
Garante a validação da coorte sem distorções de censura.

In [ ]:
df_model = df[df['ELEGIVEL'] == 1].copy()

total_elegiveis = len(df_model)
total_evasores = int((df_model['EVADIU'] == 1).sum())
taxa_evasao = df_model['EVADIU'].mean()

print(f'Total de alunos elegíveis: {total_elegiveis:,}')
print(f'Total de evasores reais:   {total_evasores:,}')
print(f'Taxa de evasão da base:    {taxa_evasao:.2%}')

## 3. Engenharia de Features Adaptada para Aperfeiçoamento
Criamos novos indicadores com foco em:
1. **Risco Financeiro e Inadimplência:** Proporção de títulos não pagos, valor em aberto e atraso máximo.
2. **Engajamento Digital no AVA:** Sinais de inatividade recente e desaceleração.
3. **Agrupamento de Cauda Longa:** Tratamento das categorias raras em `CURSO` e `END_MUNICIPIO` para evitar ruído.

In [ ]:
# 1. Tratamento e preenchimento de nulos financeiros
df_model['FIN_DEVEDOR'] = df_model['FIN_DEVEDOR'].fillna(0)
df_model['FIN_QTD_EM_ABERTO'] = df_model['FIN_QTD_EM_ABERTO'].fillna(0)
df_model['FIN_VALOR_EM_ABERTO'] = df_model['FIN_VALOR_EM_ABERTO'].fillna(0)
df_model['FIN_QTD_PAGO_ATRASO'] = df_model['FIN_QTD_PAGO_ATRASO'].fillna(0)
df_model['FIN_QTD_TITULOS'] = df_model['FIN_QTD_TITULOS'].fillna(0)
df_model['FIN_VALOR_TOTAL'] = df_model['FIN_VALOR_TOTAL'].fillna(0)

# 2. Features de Risco Financeiro
df_model['RISCO_FINANCEIRO'] = ((df_model['FIN_DEVEDOR'] == 1) | (df_model['FIN_QTD_EM_ABERTO'] > 0)).astype(int)
df_model['PROP_TITULOS_EM_ABERTO'] = df_model['FIN_QTD_EM_ABERTO'] / (df_model['FIN_QTD_TITULOS'] + 1)
df_model['PROP_TITULOS_ATRASADOS'] = df_model['FIN_QTD_PAGO_ATRASO'] / (df_model['FIN_QTD_TITULOS'] + 1)
df_model['TICKET_ALTO'] = (df_model['FIN_VALOR_TOTAL'] > df_model['FIN_VALOR_TOTAL'].median()).astype(int)
df_model['ATRASO_CRITICO'] = (df_model['FIN_MAX_DIAS_ATRASO'].fillna(0) > 15).astype(int)

# 3. Features de Engajamento no AVA
df_model['RISCO_SUMIU_AVA'] = (df_model['AVA_DIAS_DESDE_ULT_ACESSO'].fillna(999) > 30).astype(int)
df_model['RATIO_GAP_ATIVOS'] = df_model['AVA_MAIOR_GAP_DIAS'].fillna(0) / (df_model['AVA_DIAS_ATIVOS'].fillna(0) + 1)
df_model['SESSOES_POR_DIA_ATIVO'] = df_model['AVA_QTD_SESSOES'].fillna(0) / (df_model['AVA_DIAS_ATIVOS'].fillna(0) + 1)
df_model['DESACELEROU_AVA'] = (df_model['AVA_TEND_SESSOES'].fillna(0) < 0.5).astype(int)

# 4. Tratamento de cauda longa para categorias de alta cardinalidade
principais_mun = df_model['END_MUNICIPIO'].value_counts().nlargest(15).index.tolist()
df_model['END_MUNICIPIO'] = df_model['END_MUNICIPIO'].apply(lambda x: str(x) if x in principais_mun else 'OUTRO')

principais_cursos = df_model['CURSO'].value_counts().nlargest(20).index.tolist()
df_model['CURSO'] = df_model['CURSO'].apply(lambda x: str(x) if x in principais_cursos else 'OUTRO')

print('✅ Novas features de risco financeiro e engajamento criadas com sucesso!')

## 4. Separação de $X$ (Features) e $y$ (Target)
Descartamos identificadores únicos, rótulos e colunas que causariam vazamento de dados.

In [ ]:
colunas_remover = [
    'ALUNO',           # Identificador unico (vazamento/overfitting)
    'CURSOU_ALVO',      # Insumo bruto do alvo
    'SIT_ALVO',         # Derivado do alvo
    'ELEGIVEL',         # Filtro de controle
    'SIT_FIM_OBS',      # Situacao de controle
    'DT_INICIO_OBS',    # Metadado
    'DT_CORTE',         # Metadado
    'ANO_OBS',          # Metadado constante
    'PERIODO_OBS',      # Metadado constante
    'NOME_CURSO',       # Redundante com o codigo CURSO
    'FACULDADE',        # Metadado constante
    'DT_INGRESSO',      # Data exata
    'EVADIU'            # O rotulo alvo
]

X = df_model.drop(columns=[c for c in colunas_remover if c in df_model.columns])
y = df_model['EVADIU'].astype(int)

# Eliminar colunas 100% nulas desta modalidade (sem notas/faltas tradicionais)
cols_all_nan = [c for c in X.columns if X[c].isna().all()]
if cols_all_nan:
    print(f'Descartando {len(cols_all_nan)} colunas 100% nulas: {cols_all_nan}')
    X = X.drop(columns=cols_all_nan)

print(f'Total de features restantes em X: {X.shape[1]}')

## 5. Encoding de Categóricas e Preparação
Convertemos variáveis categóricas via One-Hot Encoding (`pd.get_dummies`).

In [ ]:
cat_cols = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
print(f'Colunas categóricas tratadas ({len(cat_cols)}): {cat_cols}')

X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)
print(f'Dimensões finais de X após encoding: {X_encoded.shape[1]} colunas')

## 6. Divisão Treino (80%) e Teste (20%) Estratificado
Preserva a proporção exata da classe minoritária (evasores).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f'Treino: {X_train.shape[0]:,} amostras | Evasores: {y_train.sum():,} ({y_train.mean():.2%})')
print(f'Teste:  {X_test.shape[0]:,} amostras | Evasores: {y_test.sum():,} ({y_test.mean():.2%})')

## 7. Treinamento do Modelo Random Forest (Sensível a Custo)
Utilizamos `RandomForestClassifier` com imputação pela mediana e penalização de classe desbalanceada (`scale_pos_weight`).

In [ ]:
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

# Fator de balanceamento para a classe minoritaria
scale_pos = max(1.0, (len(y_train) - y_train.sum()) / max(1, y_train.sum()))

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_split=4,
    min_samples_leaf=2,
    class_weight={0: 1.0, 1: float(scale_pos)},
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_imp, y_train)
print('✅ Modelo Random Forest treinado com sucesso!')

## 8. Otimização do Limiar de Decisão (*Threshold Tuning*)
Comparamos o desempenho do modelo variando o corte de probabilidade ($0.50 \rightarrow 0.20$).
O foco institucional é **maximizar o Recall** para que quase nenhum aluno em risco deixe de ser alertado.

In [ ]:
y_proba = rf_model.predict_proba(X_test_imp)[:, 1]

thresholds = [0.50, 0.40, 0.35, 0.30, 0.25, 0.20]
tabela_thresh = []

for t in thresholds:
    y_p = (y_proba >= t).astype(int)
    cm = confusion_matrix(y_test, y_p)
    rec = recall_score(y_test, y_p, zero_division=0)
    prec = precision_score(y_test, y_p, zero_division=0)
    f1 = f1_score(y_test, y_p, zero_division=0)
    evasores_capturados = cm[1, 1] if cm.shape == (2,2) else 0
    evasores_perdidos = cm[1, 0] if cm.shape == (2,2) else y_test.sum()
    
    tabela_thresh.append({
        'Threshold': f'{t:.2f}',
        'Recall (Sensibilidade)': f'{rec:.2%}',
        'Precision': f'{prec:.2%}',
        'F1-Score': f'{f1:.2%}',
        'Evasores Capturados': f'{evasores_capturados} / {y_test.sum()}',
        'Evasores Perdidos (FN)': evasores_perdidos
    })

df_thresh = pd.DataFrame(tabela_thresh)
print(df_thresh.to_string(index=False))

## 9. Avaliação com Limiar Otimizado (Threshold = 0.30)
No corte calibrado em **0.30**, o modelo atinge sensibilidade altíssima com excelente capacidade de ordenação na fila.

In [ ]:
OPTIMAL_THRESH = 0.30
y_pred_opt = (y_proba >= OPTIMAL_THRESH).astype(int)

print('='*65)
print(f'   RELATÓRIO DE CLASSIFICAÇÃO (THRESHOLD OTIMIZADO = {OPTIMAL_THRESH})')
print('='*65)
print(classification_report(y_test, y_pred_opt, target_names=['Permanência (0)', 'Evasão (1)'], zero_division=0))

roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)

print('='*65)
print(f'ROC-AUC Geral:                {roc_auc:.4f}')
print(f'PR-AUC (Average Precision):   {pr_auc:.4f}')
print('='*65)

# Metrica de Retencao Operacional: Top 50 da Fila (base proporcional)
k_top = min(50, len(y_test))
top_idx = np.argsort(y_proba)[::-1][:k_top]
top_reais = y_test.iloc[top_idx].sum()
recall_top = top_reais / max(1, y_test.sum())
precision_top = top_reais / k_top

print(f'Evasores reais no Top {k_top}:     {top_reais} de {y_test.sum()} evasores no teste')
print(f'Recall@{k_top}:                    {recall_top:.2%}')
print(f'Precision@{k_top}:                 {precision_top:.2%}')
print('='*65)

## 10. Matrizes de Confusão Comparativas
Comparação visual entre o limiar padrão de machine learning (0.50) e o limiar otimizado de negócio (0.30).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz 1: Threshold Padrao (0.50)
y_pred_padrao = (y_proba >= 0.50).astype(int)
cm_padrao = confusion_matrix(y_test, y_pred_padrao)
sns.heatmap(cm_padrao, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False,
            xticklabels=['Previsto: Fica', 'Previsto: Evadiu'],
            yticklabels=['Real: Fica', 'Real: Evadiu'])
axes[0].set_title(f'Limiar Padrão (0.50)\nRecall: {recall_score(y_test, y_pred_padrao, zero_division=0):.1%}', fontweight='bold')
axes[0].set_ylabel('Classe Real', fontweight='bold')
axes[0].set_xlabel('Classe Prevista', fontweight='bold')

# Matriz 2: Threshold Otimizado (0.30)
cm_opt = confusion_matrix(y_test, y_pred_opt)
sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Greens', ax=axes[1], cbar=False,
            xticklabels=['Previsto: Fica', 'Previsto: Evadiu'],
            yticklabels=['Real: Fica', 'Real: Evadiu'])
axes[1].set_title(f'Limiar Otimizado (0.30) [RECOMENDADO]\nRecall: {recall_score(y_test, y_pred_opt, zero_division=0):.1%}', fontweight='bold')
axes[1].set_ylabel('Classe Real', fontweight='bold')
axes[1].set_xlabel('Classe Prevista', fontweight='bold')

plt.tight_layout()
plt.show()

## 11. Curva ROC e Curva Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[0].plot(fpr, tpr, color='#1F4E78', lw=2.5, label=f'Random Forest (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', lw=1.5)
axes[0].set_title('Curva ROC', fontweight='bold')
axes[0].set_xlabel('Taxa de Falsos Positivos (FPR)')
axes[0].set_ylabel('Taxa de Verdadeiros Positivos (Recall / TPR)')
axes[0].legend(loc='lower right')

# Curva Precision-Recall
prec_c, rec_c, _ = precision_recall_curve(y_test, y_proba)
axes[1].plot(rec_c, prec_c, color='#009a73', lw=2.5, label=f'Random Forest (PR-AUC = {pr_auc:.3f})')
axes[1].axhline(y_test.mean(), color='crimson', linestyle='--', lw=1.5, label=f'Baseline ({y_test.mean():.1%})')
axes[1].set_title('Curva Precision-Recall', fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='upper right')

plt.tight_layout()
plt.show()

## 12. Importância das Features (*Feature Importances*)
Identifica quais variáveis exercem o maior impacto preditivo na evasão de Aperfeiçoamento.

In [ ]:
feat_df = pd.DataFrame({
    'Feature': X_encoded.columns,
    'Importancia': rf_model.feature_importances_
}).sort_values('Importancia', ascending=False)

plt.figure(figsize=(10, 8))
sns.barplot(
    data=feat_df.head(20),
    x='Importancia',
    y='Feature',
    hue='Feature',
    palette='viridis',
    legend=False
)
plt.title('Top 20 Features Mais Importantes - Aperfeiçoamento / Extensão', fontsize=13, fontweight='bold', pad=15)
plt.xlabel('Importância Relativa (Gini Importance)', fontsize=11, fontweight='bold')
plt.ylabel('Feature', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

## 13. Salvamento do Artefato em Pickle (`modelo_evasao_extensao.pkl`)
Exporta o modelo treinado, pipeline de imputação, colunas e metadados de inferência com corte calibrado em **0.30**.

In [ ]:
# Caminho de salvamento do modelo em pickle
candidate_pkl_dirs = [
    os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else '',
    'ml/extensao',
    'C:/Users/leonardo.marques/Documents/machine-learning-evasao/ml/extensao',
    '.'
]
pkl_dir = next((d for d in candidate_pkl_dirs if d and os.path.exists(d)), '.')
pkl_path = os.path.join(pkl_dir, 'modelo_evasao_extensao.pkl')

artifact = {
    'model': rf_model,
    'imputer': imputer,
    'feature_names': X_encoded.columns.tolist(),
    'threshold': 0.30,
    'principais_municipios': principais_mun,
    'principais_cursos': principais_cursos,
    'colunas_remover': colunas_remover,
    'metadata': {
        'modalidade': 'Aperfeicoamento / Extensao',
        'coorte': '2025/5',
        'n_features': X_encoded.shape[1],
        'roc_auc': float(roc_auc),
        'pr_auc': float(pr_auc),
        'recall_threshold_030': float(recall_score(y_test, y_pred_opt)),
        'precision_threshold_030': float(precision_score(y_test, y_pred_opt))
    }
}

joblib.dump(artifact, pkl_path)
print(f'🎉 Modelo salvo com sucesso em: {pkl_path}')
print(f'📦 Tamanho do arquivo: {os.path.getsize(pkl_path)/(1024*1024):.2f} MB')
print(f'🎯 Threshold configurado: {artifact["threshold"]}')
print(f'📈 Recall atingido no teste: {artifact["metadata"]["recall_threshold_030"]:.2%}')